# 06.04 · Embeddings: local vs hosted

**Module 06 · Documents, Embeddings & RAG** · Part 1 (Core)

| | |
|---|---|
| ⏱️ Estimated time | 120–150 min |
| 📋 Prerequisites | Lessons 06.02–06.03; Lesson 03.04 (HashingEmbeddings). |
| 🔑 Keys | none (`OPENAI_API_KEY` optional, for one paid comparison) |
| 🧠 Kernel | **Python (lc-mastery)** |
| ✅ Docs verified | 2026-10-12 |

> **How to use this notebook:** read each Markdown cell, then run the code cell below it with **Shift + Enter**, top to bottom.
> Cell markers: **🧪** offline (scripted model, no network, free) · **🌐** network, no tokens · **💸** calls a model (uses your free-tier quota).
> Without a key, 💸 cells print `⏭️ Skipped` and everything else still runs, so **Kernel → Restart Kernel and Run All Cells…** always completes.

## Learning objectives

1. **Explain** what an embedding is and why cosine similarity finds related text.
2. **Run** a real embedding model **locally and free** (fastembed `BAAI/bge-small-en-v1.5`), with an offline fallback.
3. **Measure** embedding models on a labelled set (recall@k, MRR) instead of trusting leaderboards.
4. **Cache** embeddings so re-running a notebook costs nothing (Jupyter skill).
5. **Compare** local options (fastembed, Ollama, sentence-transformers) with paid APIs on cost, speed, privacy and setup.

## 📚 Official reading (verified 2026-10-12)

| Page | Read it for |
|---|---|
| [Semantic search: embeddings](https://docs.langchain.com/oss/python/langchain/knowledge-base) | The `Embeddings` interface: `embed_documents`, `embed_query` |
| [FastEmbed](https://qdrant.github.io/fastembed/) | Local ONNX embeddings, no PyTorch. Used directly by `mastery.rag` (the LangChain wrapper lives in the unmaintained `langchain-community`) |
| [FastEmbed docs](https://qdrant.github.io/fastembed/) | Supported models, cache folder, re-rankers |
| [Ollama embeddings](https://docs.langchain.com/oss/python/integrations/embeddings/ollama) | Local embeddings served by Ollama |
| [Hugging Face embeddings](https://docs.langchain.com/oss/python/integrations/embeddings/huggingfacehub) | `langchain-huggingface` + sentence-transformers (needs PyTorch) |

---
## Step 0 · Install and set up

In [ ]:
%pip install --quiet "langchain-core>=1.6.9,<2" "langchain-text-splitters>=1.1.3,<2" "pypdf>=6.20.1,<7" "beautifulsoup4>=4.15.0,<5" "fastembed>=0.9.0,<1"

**You should now see** *"Note: you may need to restart the kernel to use updated packages."*, possibly below a few lines from pip. IPython prints that note after **every** `%pip`; restart only if a package was **upgraded** after you had already imported it (Lesson 00.05). `%pip` always installs into **this kernel's** environment; `!pip` might not (Lesson 00.02).

In [ ]:
from importlib import metadata

for package in ("langchain-core", "langchain-text-splitters",):
    print(f"{package:<22} {metadata.version(package)}")

In [ ]:
# Step 1: Find the repository root (Jupyter starts in modules/06-rag/).
import importlib.util
import sys
from pathlib import Path

REPO_ROOT = next(
    (p for p in (Path.cwd(), *Path.cwd().parents)
     if (p / "pyproject.toml").is_file() and (p / "src" / "mastery").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find the course repo. Start Jupyter inside your clone.")

# Step 2: Import `mastery` from THIS clone's src/, like pytest does (pythonpath = ["src"]), and check
# that the env's own install (00.02 guide, Step 5) points here too: terminal commands use that one.
SRC_DIR = (REPO_ROOT / "src").resolve()
installed_spec = importlib.util.find_spec("mastery")
if sys.path[0] != str(SRC_DIR):
    sys.path.insert(0, str(SRC_DIR))
import mastery

print("Repo root:", REPO_ROOT)
print("mastery  :", mastery.__version__, "from", mastery.__file__)
installed_at = installed_spec.origin if installed_spec else None
if not Path(mastery.__file__).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This kernel had already imported mastery from another folder: restart the kernel, run from the top.")
elif installed_at is None or not Path(installed_at).resolve().is_relative_to(SRC_DIR):
    print("⚠️ This env's own mastery install does not use this clone's src/:", installed_at or "(not installed)")
    print("   This notebook uses this clone's src/ anyway, but terminal commands (python -m mastery...) do not.")
    print('   Fix once (Git Bash, course env active, root of THIS clone): python -m pip install -e ".[dev]"')

In [ ]:
KEYS_TO_CHECK = ('GROQ_API_KEY', 'OPENAI_API_KEY')
# Step 3: Load .env, check keys WITHOUT printing them, read model IDs from config.
from mastery.config import env_status, is_set, load_env, load_settings

dotenv_path = load_env()
settings = load_settings(load_dotenv_file=False)   # .env was just loaded.
HAVE_GROQ_KEY = is_set("GROQ_API_KEY")

print(".env      :", dotenv_path or "not found (fine if keys are set in your shell)")
for name, status in env_status(*KEYS_TO_CHECK).items():
    print(f"{name:<18}: {status}")
print("fast      :", settings.model_fast)
print("reasoning :", settings.model_reasoning)
if not HAVE_GROQ_KEY:
    print("\n⚠️ No GROQ_API_KEY: 💸 cells will print ⏭️ Skipped; 🧪 cells still teach the concept.")

In [ ]:
# Step 4: Silence known, harmless noise so real problems stand out. Set these BEFORE importing chromadb/fastembed.
import os
import warnings

os.environ.setdefault("ANONYMIZED_TELEMETRY", "False")          # Chroma: no usage telemetry.
os.environ.setdefault("HF_HUB_DISABLE_SYMLINKS_WARNING", "1")   # Windows: HF cache can't symlink without Developer Mode.
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")        # HF tokenizers' fork warning in Jupyter.
# Same as Lesson 02.04: langchain-core 1.6.9/1.6.10 Pydantic serializer warning about the injected runtime.
warnings.filterwarnings("ignore", message="Pydantic serializer warnings", category=UserWarning)
# tqdm (used by the model downloader) asks for ipywidgets to draw a progress bar; text output is fine.
warnings.filterwarnings("ignore", message="IProgress not found")

In [ ]:
# Step 5: The course corpus: 24 Markdown files + 8 PDFs (23 pages) about fictional Nimbus weather stations.
import json

RAG_DIR = REPO_ROOT / "data" / "samples" / "rag"
CORPUS_DIR = RAG_DIR / "corpus"
GOLDEN = [json.loads(line) for line in (RAG_DIR / "golden-set.jsonl").read_text(encoding="utf-8").splitlines()]
print("corpus files:", sum(1 for p in CORPUS_DIR.rglob("*") if p.is_file()), "| golden questions:", len(GOLDEN))

In [ ]:
# Step 6: Corpus → chunks (06.02–06.03), the labelled paraphrase set, and a cosine helper.
import math

from mastery.config import is_set
from mastery.rag import load_corpus, split_documents

chunks = split_documents(load_corpus(CORPUS_DIR), "markdown", 800, 100)
PAIRS = [json.loads(line) for line in (RAG_DIR / "embedding-pairs.jsonl").read_text(encoding="utf-8").splitlines()]
PARAPHRASE_SET = [{"id": f"p{i:02d}", "question": p["question"], "expected_sources": [p["expected_source"]],
                   "in_corpus": True} for i, p in enumerate(PAIRS, start=1)]
HAVE_OPENAI_KEY = is_set("OPENAI_API_KEY")

def cosine(a: list[float], b: list[float]) -> float:
    dot = sum(x * y for x, y in zip(a, b))
    return dot / ((math.sqrt(sum(x * x for x in a)) * math.sqrt(sum(y * y for y in b))) or 1.0)

print(len(chunks), "chunks |", len(PARAPHRASE_SET), "paraphrase questions, e.g.:", PAIRS[0])

**You should now see** `73 chunks | 10 paraphrase questions, e.g.: {'question': 'How many weeks can I wait before sending a purchase back?', 'expected_source': 'policies/returns.md'}`.

The **paraphrase set** is the interesting one: its questions use **different words** than the documents ("sending a purchase back" vs "returns"). That's what real users do.

---
## Part 1 · What is an embedding?

An embedding model turns text into a vector (a list of numbers, e.g. 384 of them) such that texts with similar **meaning** get vectors pointing in similar directions. **Cosine similarity** measures that direction: 1 = same, 0 = unrelated. Let's start with the offline `HashingEmbeddings` from Module 03: it hashes **words**, so it only sees shared words.

In [ ]:
# 🧪 Offline: word-based "embeddings" see shared words, not meaning.
from mastery.memory import HashingEmbeddings

SENTENCES = ["You can return a product within 30 days.",
             "How many weeks can I wait before sending a purchase back?",
             "The Nimbus Pro has a wind sensor."]

def similarity_table(embeddings, label: str) -> None:
    vectors = embeddings.embed_documents(SENTENCES)
    print(f"{label} (dims={len(vectors[0])})")
    for i in range(len(SENTENCES)):
        print("  ", " ".join(f"{cosine(vectors[i], vectors[j]):5.2f}" for j in range(len(SENTENCES))), "|", SENTENCES[i][:45])

hashing = HashingEmbeddings(dims=1024)
similarity_table(hashing, "HashingEmbeddings")

**You should now see** a 3×3 table with 1.00 on the diagonal. Sentences 1 and 2 mean nearly the same thing but share almost no words, so their similarity is low (≈ 0.1: only "can" matches after stop words). Word matching can't find that pair.

---
## Part 2 · A real local model: fastembed `bge-small-en-v1.5`

fastembed runs small ONNX models **on your CPU**: no PyTorch, no GPU, no key, free. `BAAI/bge-small-en-v1.5` gives 384-dimensional vectors and is a strong small English model. `mastery.memory.local_embeddings()` caches the model in `~/.cache/fastembed` (fastembed's default is the temp folder, which Windows cleans up).

> 🌐 First run downloads a small model from Hugging Face (no key, no tokens), cached afterwards. Offline, blocked, or `[embeddings]` not installed? The cell prints a ⚠️ and the lesson continues with the offline `HashingEmbeddings`.

In [ ]:
# 🌐 First run downloads ~67 MB from Hugging Face; afterwards it loads from the cache (offline OK).
try:
    from mastery.memory import local_embeddings

    local = local_embeddings()          # BAAI/bge-small-en-v1.5
    LOCAL_NAME = local.model_name
except Exception as exc:                # Extra not installed, offline, proxy/SSL...
    print(f"⚠️ Local embeddings unavailable ({type(exc).__name__}: {str(exc)[:150]})")
    print("   Fix: python -m pip install -e \".[dev,embeddings]\" (and the SSL fix in 00.02 Step 4b). Using HashingEmbeddings.")
    local, LOCAL_NAME = HashingEmbeddings(dims=1024), "HashingEmbeddings (fallback)"
similarity_table(local, LOCAL_NAME)

**You should now see** `BAAI/bge-small-en-v1.5 (dims=384)` and a table where sentences 1 and 2 are clearly **more similar to each other** (≈ 0.7) than to sentence 3 (≈ 0.4–0.5). Real embeddings have a "floor" (unrelated sentences rarely go near 0), so compare **relative** values, not absolute ones. (Offline you see the fallback table again.)

---
## Part 3 · Measure on a labelled set

Leaderboards rank models on other people's data. What matters is **your** questions and **your** documents. We index the corpus with each model and check whether the expected file comes back (06.09 explains the metrics in detail).

In [ ]:
# 🧪/🌐 Measure: golden set (shares words with the docs) and paraphrase set (doesn't).
import time

from mastery.rag import build_index, evaluate_retrieval, retrieve

def measure(embeddings, label: str) -> dict:
    start = time.perf_counter()
    store = build_index(chunks, embeddings)
    seconds = time.perf_counter() - start
    row = {"model": label, "index_s": round(seconds, 2)}
    for name, questions in [("golden", GOLDEN), ("paraphrase", PARAPHRASE_SET)]:
        r1 = evaluate_retrieval(lambda q: retrieve(store, q, k=1), questions, k=1)
        r5 = evaluate_retrieval(lambda q: retrieve(store, q, k=5), questions, k=5)
        row |= {f"{name} R@1": r1["recall"], f"{name} R@5": r5["recall"], f"{name} MRR": r5["mrr"]}
    return row

results = [measure(hashing, "HashingEmbeddings"), measure(local, LOCAL_NAME)]
for row in results:
    print(row)

**You should now see** two rows. On our run:

| model | golden R@1 | golden R@5 | paraphrase R@1 | paraphrase R@5 | paraphrase MRR | index time |
|---|---|---|---|---|---|---|
| HashingEmbeddings | 0.72 | 1.00 | 0.50 | 0.80 | 0.61 | ~0.01 s |
| bge-small-en-v1.5 | 0.91 | 1.00 | 0.80 | 1.00 | 0.88 | ~1 s (CPU) |

On word-friendly questions both find the file in the top 5; on **paraphrases** the real model wins clearly, and it puts the right file **first** far more often. That's the gap meaning makes. Your exact numbers may differ slightly (CPU, versions).

### What just happened?

Same chunks, same questions, only the embedding model changed. A 10-question labelled set took two minutes to write and already separates the models. Do this before paying for a hosted model: on a small, clean English corpus a free local model is often enough.

---
## Part 4 · Jupyter skill: cache your embeddings

Every "Run All" re-embeds every chunk. Locally that costs time; with a paid API it costs money. `CachedEmbeddings` wraps any model with a SQLite cache keyed by **model name + text**, stored in `data/cache/embeddings/` (git-ignored). Change the model and the key changes, so vectors from two models never mix.

In [ ]:
%%time
# 🧪/🌐 First run: computes and stores every vector (the cache may already be warm from an earlier run).
from mastery.rag import CachedEmbeddings

cached = CachedEmbeddings(local)
vectors = cached.embed_documents([c.page_content for c in chunks])
print(f"{len(vectors)} vectors | hits={cached.hits} misses={cached.misses} | cache file: {cached.path.relative_to(REPO_ROOT).as_posix()}")

In [ ]:
%%time
# 🧪 Second run: everything comes from the cache.
vectors = cached.embed_documents([c.page_content for c in chunks])
print(f"hits={cached.hits} misses={cached.misses}")

**You should now see** the first cell report `misses=73` on a fresh cache (or `hits=73` if you ran it before) and a wall time around a second with the real model; the second cell adds 73 **hits** and takes a few milliseconds. `%%time` (first line of a cell) prints CPU and wall time for the whole cell; `%time` times one line.

Where things go (check with `git check-ignore -v <path>` in Git Bash: a matching rule means git ignores it):

| What | Where | In git? |
|---|---|---|
| Embedding model files | `~/.cache/fastembed` (`FASTEMBED_CACHE_PATH` to move it) | No (outside the repo) |
| Embedding cache | `data/cache/embeddings/*.sqlite` | No (`data/cache/`) |
| Vector stores (06.05) | `data/vectorstores/` | No |

Queries are not cached (`embed_query`): they're cheap and rarely repeat.

---
## Part 5 · Other options: Ollama, sentence-transformers, paid APIs

| Option | Setup | Cost | Privacy | Notes |
|---|---|---|---|---|
| **fastembed** (this course) | `pip` only, ONNX, CPU | Free | Local | Small, fast; ~67 MB model |
| **Ollama** (`langchain-ollama` 1.1.0) | Install [Ollama](https://ollama.com), `ollama pull nomic-embed-text` | Free | Local | Good if you already run Ollama (Module 14) |
| **sentence-transformers** (`langchain-huggingface` 1.2.3) | Needs **PyTorch** (~200 MB CPU wheel, more with CUDA) | Free | Local | Thousands of models; heavier install |
| **OpenAI** `text-embedding-3-small` | `OPENAI_API_KEY` | US$0.02 per 1M tokens (our corpus ≈ 4k tokens) | Data leaves your machine | No local CPU cost, strong multilingual |
| Cohere, Voyage, Google, Mistral... | API key | Paid (some free tiers) | Data leaves your machine | Compare on **your** labelled set |

The next cell tries the optional local ones only if they're already there; nothing is installed for you.

In [ ]:
# 🧪/🌐 Optional local alternatives: used only if present (no installs, no errors).
import importlib.util

import httpx

try:
    ollama_up = httpx.get("http://127.0.0.1:11434/api/tags", timeout=1).status_code == 200
except httpx.HTTPError:
    ollama_up = False
if ollama_up and importlib.util.find_spec("langchain_ollama"):
    from langchain_ollama import OllamaEmbeddings

    try:
        results.append(measure(OllamaEmbeddings(model="nomic-embed-text"), "ollama nomic-embed-text"))
        print(results[-1])
    except Exception as exc:
        print(f"⚠️ Ollama is running but the model failed ({str(exc)[:100]}). Try: ollama pull nomic-embed-text")
else:
    print("ℹ️ Ollama: not running on 127.0.0.1:11434 (or langchain-ollama not installed): skipped.")
print("ℹ️ sentence-transformers:", "installed" if importlib.util.find_spec("sentence_transformers") else
      "not installed (needs PyTorch; optional): skipped.")

**You should now see** two `ℹ️ ... skipped` lines (unless you run Ollama). With Ollama running and the model pulled, a third row appears in `results`.

In [ ]:
# 💸 Paid: OpenAI text-embedding-3-small on the same sets (≈ 5k tokens ≈ US$0.0001). Needs OPENAI_API_KEY.
if HAVE_OPENAI_KEY:
    from langchain_openai import OpenAIEmbeddings

    results.append(measure(CachedEmbeddings(OpenAIEmbeddings(model="text-embedding-3-small"),
                                            namespace="openai-text-embedding-3-small"), "openai 3-small"))
    print(results[-1])
else:
    print("⏭️ Skipped: OPENAI_API_KEY is not set (optional; the free local model is the course default).")

**You should now see** `⏭️ Skipped` (no OpenAI key), or a third row. Wrapping the paid model in `CachedEmbeddings` means you pay for each chunk **once**, even across kernel restarts.

---
## ⚠️ Pitfalls and troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| `ModuleNotFoundError: fastembed` | `[embeddings]` extra not installed | `python -m pip install -e ".[dev,embeddings]"` |
| `SSLError` / `CERTIFICATE_VERIFY_FAILED` downloading the model | Corporate proxy or Anaconda certificates | [00.02 Step 4b](../00-foundations/00-02-create-course-env.md#troubleshooting) (`SSL_CERT_FILE`); then rerun |
| `ImportError: DLL load failed` (onnxruntime) on Windows | Missing Visual C++ runtime | Install the [VC++ Redistributable (x64)](https://learn.microsoft.com/cpp/windows/latest-supported-vc-redist), restart Git Bash |
| Model re-downloads every few days | fastembed default cache is `%TEMP%` | `local_embeddings()` uses `~/.cache/fastembed`; or set `FASTEMBED_CACHE_PATH` |
| `UserWarning: ... symlinks` (Hugging Face) on Windows | Symlinks need Developer Mode | Harmless; Step 4 sets `HF_HUB_DISABLE_SYMLINKS_WARNING=1` |
| `OSError: [WinError 206]` / path too long | Deep model-cache paths + repo in a long folder | Short repo path (`C:\dev\...`) or enable long paths in Windows |
| Search results are nonsense after switching models | Old vectors from another model in the store | Rebuild the index; never mix models in one collection |

## ⚠️ Production notes

> ⚠️ **Production note: the embedding model is part of the index.**
> Store the model name (and version) with the index; a query embedded with model B against vectors from model A returns garbage without any error. Re-embed everything when you change models.

> ⚠️ **Production note: hosted embeddings send your documents out.**
> Check the provider's data-use and retention terms before embedding private documents; local models keep data on your machine.

## 🏋️ Exercises

1. **(Easy)** Add a fourth sentence to `SENTENCES` ("Can I send my order back after three weeks?") and compare both tables.
2. **(Medium)** Look at `measure()`'s paraphrase rows for both models: which questions does HashingEmbeddings miss? (`evaluate_retrieval(...)["rows"]`).
3. **(Medium)** Delete `data/cache/embeddings/` (kernel restart first on Windows) and re-run Part 4. What changes?
4. **(Stretch)** Try another fastembed model, e.g. `local_embeddings("BAAI/bge-base-en-v1.5")` (~210 MB). Is it better on the paraphrase set? Is it worth 3× the size?

<details>
<summary><b>Show solutions</b></summary>

**1.** The new sentence shares "send" and "order" with nothing in sentence 1; with Hashing it's close to 0 against sentence 1, with bge-small clearly higher.

**2.** `rows = evaluate_retrieval(lambda q: retrieve(build_index(chunks, hashing), q, k=5), PARAPHRASE_SET)["rows"]; [r for r in rows if not r["hit"]]`: the questions with no shared content words.

**3.** The first cell reports 73 misses again (and takes longer); the cache file is recreated.

**4.** Usually slightly better MRR, ~3× slower indexing and a bigger download. On this corpus bge-small is enough: measure, then decide.
</details>

## ✅ Checkpoint

- [ ] I can explain embeddings and cosine similarity in two sentences.
- [ ] I ran a local embedding model for free (or saw the fallback and know how to fix it).
- [ ] I compared models on a labelled set, not on a leaderboard.
- [ ] I cache embeddings and know where caches and models live (and that git ignores them).
- [ ] Restart & Run All passes; outputs cleared before saving.

## 🔎 Research checkpoint

Which embedding models does fastembed support today, and is `BAAI/bge-small-en-v1.5` still its default? See the [FastEmbed docs](https://qdrant.github.io/fastembed/) (or `TextEmbedding.list_supported_models()`). Log the answer in [`docs/research-log.md`](../../docs/research-log.md).

## Summary

- Embeddings map meaning to direction; cosine similarity compares directions.
- A free local model (fastembed, CPU) beats word matching on paraphrases; measure on your own labelled set.
- Cache embeddings by model + text; keep caches and models out of git.
- Paid models: compare first, cache always, check data terms.

---
## ➡️ What's next

**You're done with this lesson when** every box in the ✅ Checkpoint is ticked and **Kernel → Restart Kernel and Run All Cells…** finishes without errors.

1. Clear outputs before you save (**Edit → Clear Outputs of All Cells**), then **File → Save** (`Ctrl + S`).
2. Optional: log your research checkpoint in [`docs/research-log.md`](../../docs/research-log.md).
3. Next: **Lesson 06.05 · Vector stores: in-memory → Chroma → production options** → [`06-05-vector-stores.ipynb`](06-05-vector-stores.ipynb).

[Module 06 README](README.md) · [Course outline](../../COURSE_OUTLINE.md)